# R0 — Contrato de datos y split canónico

**Primer notebook del pipeline. Todo lo demás depende de este**

## Qué corrige

| Problema de la v1 | Corrección aquí |
|---|---|
| El balanceo y CTGAN se aplicaban al dataset completo; el holdout se extraía después | El split se hace **primero**, antes de cualquier transformación |
| La identidad de fila se perdía al guardar con `index=False`, de modo que `drop(index=...)` eliminaba filas equivocadas | `row_id` es una **columna real** que viene en el CSV y viaja en todos los archivos |
| Split por fila, con sesiones del mismo cliente a ambos lados | Split **agrupado por `customer_id`** |
| `session_duration_s` constante y duplicados exactos usados como features | Auditoría de columnas y **contrato de features** calculado solo sobre train |
| Dos datasets sintéticos distintos mezclados entre fases | Un único dataset canónico: `biocatch_sinthetic_data_v3.csv` |


Otros cambios de esquema que afectan a este notebook:

* Ya no hay `device_type` ni filtro por canal: el dataset es 100 % móvil.
* `transaction_amount_cop` y `time_to_transaction_s` usan **-1 como centinela**
  de "no aplica". No es un cero ni un faltante, y ningún paso posterior puede
  tratarlo como tal.
* `claim_category` usa la cadena vacía, que `pd.read_csv` convertiría en NaN:
  por eso se carga con `vc.read_raw()` y no con `vc.read_csv()`.

## Requisito previo

Subir el dataset canónico a la ruta que imprime la celda siguiente.
Es **el único archivo de entrada** del pipeline.

In [1]:
# Celda de arranque idéntica en todos los notebooks. Deja el kernel de SageMaker
# en un estado conocido: mismo directorio de trabajo, mismas semillas, mismas
# versiones. Si algo de esto cambia entre corridas, los resultados no son
# comparables aunque los notebooks sean los mismos.
import sys, os

AQUI = os.getcwd()                    # los notebooks y vishing_common.py conviven
if AQUI not in sys.path:
    sys.path.insert(0, AQUI)

import numpy as np
import pandas as pd
import vishing_common as vc

vc.set_all_seeds()                    # random, numpy y torch (+ cuDNN determinista)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

print("dataset:", vc.RAW_FILENAME, "(esquema", vc.DATASET_VERSION + ")")
print("bucket :", vc.BUCKET, "| prefijo:", vc.PREFIX)
print("seed   :", vc.SEED, "| split:", vc.SPLIT_MODE, "| política:", vc.FEATURE_POLICY)
print("xgboost se ejecutará en:", vc.xgb_device())
vc.check_versions()

dataset: biocatch_sinthetic_data_v3.csv (esquema v3)
bucket : poc-vishing | prefijo: v2
seed   : 42 | split: grouped | política: audited
xgboost se ejecutará en: cuda
  versiones OK: numpy 2.0.2, pandas 2.2.3, scipy 1.14.1, sklearn 1.5.2, imblearn 0.12.4, xgboost 2.1.4


,paquete,instalada,esperado,ok
0,numpy,2.0.2,">=1.26,<2.1",True
1,pandas,2.2.3,">=2.1,<2.3",True
2,scipy,1.14.1,">=1.11,<1.15",True
3,sklearn,1.5.2,">=1.4,<1.6",True
4,imblearn,0.12.4,">=0.12,<0.13",True
5,xgboost,2.1.4,">=2.0,<2.2",True


In [2]:
print("Único archivo que debes subir a S3:")
print("   ", vc.P.raw)
print()
print("Este notebook escribirá:")
for k in ["audit_report", "feature_contract", "run_manifest",
          "split_map", "train", "val", "test"]:
    print("   ", getattr(vc.P, k))

Único archivo que debes subir a S3:
    s3://poc-vishing/v2/00_raw/biocatch_sinthetic_data_v3.csv

Este notebook escribirá:
    s3://poc-vishing/v2/01_contract/data_audit.csv
    s3://poc-vishing/v2/01_contract/feature_contract.json
    s3://poc-vishing/v2/01_contract/run_manifest.json
    s3://poc-vishing/v2/02_splits/split_map.parquet
    s3://poc-vishing/v2/02_splits/train.parquet
    s3://poc-vishing/v2/02_splits/val.parquet
    s3://poc-vishing/v2/02_splits/test.parquet


## 1. Carga y validación del dataset canónico

`validate_raw` comprueba forma, balance exacto, unicidad de `row_id` y las tres
variables que estaban degeneradas. Falla aquí, no en R4.

In [3]:
df = vc.read_raw(vc.P.raw)
print("crudo:", df.shape)

integridad = vc.validate_raw(df, strict=True)
print()
print("estado de las tres variables del bug histórico:")
for k, v in integridad["no_degeneradas"].items():
    print("   %-28s %s" % (k, v))

print()
print("tasa de vishing: %.4f" % df[vc.TARGET].mean())
print("clientes únicos:", df[vc.GROUP_COL].nunique())
print("sesiones por cliente: %.2f" % (len(df) / df[vc.GROUP_COL].nunique()))

crudo: (100000, 58)
  OK dataset v3: 100,000 filas x 58 cols, 5,000 vishing, 38,000 clientes, 0 nulos

estado de las tres variables del bug histórico:
   session_duration_s           {'sd': 141.337, 'nunique': 36380}
   call_overlap_duration_s      {'corr_con_phone_call_active': 0.5987}
   time_to_transaction_s        {'corr_con_transaction_attempted': 0.6543}

tasa de vishing: 0.0500
clientes únicos: 38000
sesiones por cliente: 2.63


## 2. Split canónico — antes de tocar nada

Se reparte por **cliente**, no por fila. Con 2,63 sesiones por cliente y el
55 % de la varianza conductual atribuible al cliente (el generador v3 asigna a
cada `customer_id` rasgos latentes estables), un reparto por fila deja sesiones
del mismo cliente en train y en test, lo que infla el desempeño sin que ninguna
aserción de `row_id` lo detecte.

El conjunto de **test no se vuelve a abrir hasta R5**.

In [4]:
df = vc.make_split(df, mode=vc.SPLIT_MODE, fractions=vc.SPLIT_FRACTIONS, seed=vc.SEED)

resumen = vc.split_summary(df)
display(resumen)

# Ningún cliente puede aparecer en dos particiones
cruce = df.groupby(vc.GROUP_COL)["split"].nunique()
assert cruce.max() == 1, "hay clientes repartidos entre particiones"
print("OK: ningún cliente cruza particiones")

,sesiones,vishing,tasa_vishing,clientes
split,,,,
train,60134,3009,0.050038,22800
val,19673,999,0.050780,7600
test,20193,992,0.049126,7600


OK: ningún cliente cruza particiones


## 3. Auditoría de columnas — sobre train únicamente

Mirar el dataset completo para decidir qué variables descartar sería una forma
de *peeking*. La auditoría se calcula sobre train y se aplica a las tres
particiones.

In [5]:
train = df[df.split == "train"].copy()

descartes = set(vc.ID_COLS + vc.LEAKAGE_COLS + vc.POSTHOC_COLS + vc.REDUNDANT_V1
                + [vc.TARGET, vc.ROW_ID, vc.ORIGIN, "split"])
candidatas = [c for c in df.columns if c not in descartes]

audit = vc.audit_columns(train, candidatas)
audit = audit.sort_values(["constante", "duplicado_de", "corr_max"],
                          ascending=[False, False, False])
display(audit)

,feature,dtype,nunique,min,max,std,rango,constante,binaria,dispersion_rel,duplicado_de,corr_max,corr_max_con
18,dead_time_periods,int64,12,0.0000,1.100000e+01,1.140097e+00,1.100000e+01,False,False,0.103645,,0.8534,total_dead_time_s
19,total_dead_time_s,float64,5502,0.0000,1.469900e+02,1.338437e+01,1.469900e+02,False,False,0.091056,,0.8534,dead_time_periods
24,input_error_count,int64,12,0.0000,1.200000e+01,1.026956e+00,1.200000e+01,False,False,0.085580,,0.7371,input_correction_count
25,input_correction_count,int64,18,0.0000,2.000000e+01,1.527392e+00,2.000000e+01,False,False,0.076370,,0.7371,input_error_count
16,avg_hesitation_duration_s,float64,855,0.0000,1.337000e+01,1.350943e+00,1.337000e+01,False,False,0.101043,,0.7360,max_hesitation_duration_s
17,max_hesitation_duration_s,float64,1602,0.0000,2.949000e+01,2.726050e+00,2.949000e+01,False,False,0.092440,,0.7360,avg_hesitation_duration_s
20,dead_time_ratio,float64,4713,0.0000,9.200000e-01,1.118016e-01,9.200000e-01,False,False,0.121523,,0.7128,total_dead_time_s
42,errors_per_minute,float64,13270,0.0000,1.339960e+01,5.611985e-01,1.339960e+01,False,False,0.041882,,0.6965,input_error_count
15,hesitation_count,int64,15,0.0000,1.500000e+01,1.390431e+00,1.500000e+01,False,False,0.092695,,0.6791,max_hesitation_duration_s
38,transaction_attempted,int64,2,0.0000,1.000000e+00,4.866363e-01,1.000000e+00,False,True,0.486636,,0.6544,time_to_transaction_s


In [6]:
print("=== CONSTANTES (cero información) ===")
c = audit[audit.constante]
print(c[["feature", "nunique", "min", "max"]].to_string(index=False) if len(c) else "  ninguna")

print()
print("=== DUPLICADOS EXACTOS (|r| >= 0.9999) ===")
d = audit[audit.duplicado_de != ""]
print(d[["feature", "duplicado_de"]].to_string(index=False) if len(d) else "  ninguno")

print()
print("=== PARES MÁS CORRELACIONADOS (margen frente al umbral de duplicado) ===")
print(audit.nlargest(6, "corr_max")[["feature", "corr_max_con", "corr_max"]]
      .to_string(index=False))

print()
print("=== DISPERSIÓN RELATIVA DESPRECIABLE (std/rango < 0.02, no binarias) ===")
n = audit[(~audit.binaria) & (audit.dispersion_rel < 0.02) & (~audit.constante)]
print(n[["feature", "nunique", "min", "max", "std", "dispersion_rel"]].to_string(index=False)
      if len(n) else "  ninguna")

=== CONSTANTES (cero información) ===
  ninguna

=== DUPLICADOS EXACTOS (|r| >= 0.9999) ===
  ninguno

=== PARES MÁS CORRELACIONADOS (margen frente al umbral de duplicado) ===
                  feature              corr_max_con  corr_max
        dead_time_periods         total_dead_time_s    0.8534
        total_dead_time_s         dead_time_periods    0.8534
        input_error_count    input_correction_count    0.7371
   input_correction_count         input_error_count    0.7371
avg_hesitation_duration_s max_hesitation_duration_s    0.7360
max_hesitation_duration_s avg_hesitation_duration_s    0.7360

=== DISPERSIÓN RELATIVA DESPRECIABLE (std/rango < 0.02, no binarias) ===
  ninguna


In [ ]:
assert int(audit.constante.sum()) == 0, \
    "hay columnas constantes: el CSV no es el v3 o se corrompió al subirlo"
assert int((audit.duplicado_de != "").sum()) == 0, \
    "hay duplicados exactos: el CSV no es el v3"
print("OK: ninguna columna constante ni duplicada. Correlación máxima entre")
print("    features: %.4f (%s ~ %s), muy por debajo del umbral 0.9999."
      % (audit.corr_max.max(),
         audit.loc[audit.corr_max.idxmax(), "feature"],
         audit.loc[audit.corr_max.idxmax(), "corr_max_con"]))

OK: ninguna columna constante ni duplicada. Correlación máxima entre
    features: 0.8534 (dead_time_periods ~ total_dead_time_s), muy por debajo del umbral 0.9999.


## 4. Contrato de features

* `legacy` — todas las candidatas, sin filtrar (44 sobre el esquema v3).
* `audited` — **por defecto**. Quita constantes y duplicados exactos. Sobre el
  v3 no quita nada, y ese es el resultado esperado.
* `strict` — además quita las tres derivadas de `session_duration_s`. Ojo: el
  motivo cambió. Antes se quitaban porque la duración era constante y las
  derivadas quedaban distorsionadas; ahora la duración es informativa y el
  único argumento es la colinealidad con su numerador. `strict` pasa de ser la
  política "correcta" a ser una alternativa que conviene contrastar.

In [8]:
contratos = {p: vc.build_feature_contract(train, policy=p)
             for p in ["legacy", "audited", "strict"]}

for p, c in contratos.items():
    print("%-8s -> %2d features" % (p, c["n_features"]))
    for f, motivo in c["removed"].items():
        print("            - %-32s %s" % (f, motivo))
    print()

contract = contratos[vc.FEATURE_POLICY]
print("política activa:", vc.FEATURE_POLICY, "->", contract["n_features"], "features")

legacy   -> 44 features

audited  -> 44 features

strict   -> 41 features
            - errors_per_minute                derivada aritmética de session_duration_s (colineal con su numerador)
            - hesitation_composite             derivada aritmética de session_duration_s (colineal con su numerador)
            - dead_time_ratio                  derivada aritmética de session_duration_s (colineal con su numerador)

política activa: audited -> 44 features


In [9]:
print("=== features por grupo funcional ===")
for g, fs in contract["groups"].items():
    print("  %-12s (%d): %s" % (g, len(fs), ", ".join(fs) if fs else "-"))

print()
print("=== familias para la ablation de R6 ===")
for fam, fs in contract["families"].items():
    print("  %-12s: %d features" % (fam, len(fs)))

=== features por grupo funcional ===
  keystroke    (5): avg_keyhold_ms, avg_interkey_latency_ms, typing_speed_cps, keystroke_variability, segmented_typing_ratio
  touch        (5): avg_touch_pressure, avg_touch_size_px, swipe_speed_px_s, swipe_directional_variance, scroll_speed_avg
  motion       (5): device_tilt_angle_mean, device_tilt_variability, gyro_rotation_rate_mean, accelerometer_jerk_mean, phone_motion_events
  hesitation   (3): avg_hesitation_duration_s, max_hesitation_duration_s, hesitation_count
  dead_time    (3): total_dead_time_s, dead_time_ratio, dead_time_periods
  navigation   (3): unique_screens_visited, navigation_back_count, screen_transition_time_avg_s
  corrections  (5): input_error_count, input_correction_count, amount_field_corrections, beneficiary_field_corrections, copy_paste_events
  context      (7): session_duration_s, hour_of_day, is_atypical_hour, phone_call_active, call_overlap_duration_s, remote_access_tool_detected, suspicious_app_detected
  transact

## 5. Persistencia y aserciones

In [10]:
train = df[df.split == "train"].copy()
val   = df[df.split == "val"].copy()
test  = df[df.split == "test"].copy()

vc.assert_contract(train, contract, "train")
vc.assert_contract(val, contract, "val")
vc.assert_contract(test, contract, "test")

vc.assert_disjoint(train, val,  "train vs val")
vc.assert_disjoint(train, test, "train vs test")
vc.assert_disjoint(val,   test, "val vs test")

  OK contrato [train]: 44 features presentes
  OK contrato [val]: 44 features presentes
  OK contrato [test]: 44 features presentes
  OK sin fuga [train vs val]: 60,134 train / 19,673 eval, disjuntos
  OK sin fuga [train vs test]: 60,134 train / 20,193 eval, disjuntos
  OK sin fuga [val vs test]: 19,673 train / 20,193 eval, disjuntos


In [11]:
vc.write_parquet(df[[vc.ROW_ID, vc.GROUP_COL, "split", vc.TARGET]], vc.P.split_map)
vc.write_parquet(train, vc.P.train)
vc.write_parquet(val,   vc.P.val)
vc.write_parquet(test,  vc.P.test)

vc.write_csv(audit, vc.P.audit_report)
vc.write_json({"activo": vc.FEATURE_POLICY, "contratos": contratos}, vc.P.feature_contract)

manifiesto = vc.env_manifest()
manifiesto["r0"] = {
    "dataset": vc.RAW_FILENAME,
    "integridad_crudo": integridad,
    "sesiones": int(len(df)),
    "clientes": int(df[vc.GROUP_COL].nunique()),
    "split": {k: int(v) for k, v in df.split.value_counts().items()},
    "clientes_por_split": {s: int(df[df.split == s][vc.GROUP_COL].nunique())
                           for s in ["train", "val", "test"]},
    "tasa_vishing": {s: float(df[df.split == s][vc.TARGET].mean())
                     for s in ["train", "val", "test"]},
    "n_features": contract["n_features"],
    "features_descartadas": contract["removed"],
    "corr_max_entre_features": float(audit.corr_max.max()),
}
vc.write_json(manifiesto, vc.P.run_manifest)
print()
print("Este manifiesto es el que debe citarse en el paper: contiene las")
print("versiones EXACTAS que resolvió esta corrida, no los rangos declarados.")
print()
print("R0 completo. Continuar con R1 (EDA sobre train).")

  escrito s3://poc-vishing/v2/02_splits/split_map.parquet  (100,000 filas x 4 cols)
  escrito s3://poc-vishing/v2/02_splits/train.parquet  (60,134 filas x 60 cols)
  escrito s3://poc-vishing/v2/02_splits/val.parquet  (19,673 filas x 60 cols)
  escrito s3://poc-vishing/v2/02_splits/test.parquet  (20,193 filas x 60 cols)
  escrito s3://poc-vishing/v2/01_contract/data_audit.csv  (44 filas)
  escrito s3://poc-vishing/v2/01_contract/feature_contract.json
  escrito s3://poc-vishing/v2/01_contract/run_manifest.json

Este manifiesto es el que debe citarse en el paper: contiene las
versiones EXACTAS que resolvió esta corrida, no los rangos declarados.

R0 completo. Continuar con R1 (EDA sobre train).
